# sequence_adapter_batch_v1

验证第一批 sequence 模型 adapter：`sequence_lag_ridge_mse`、`lstm_mse`、`gru_mse`。本轮目标是把序列窗口构造、PyTorch recurrent estimator、artifact 保存/加载和 matrix analysis 链路接起来。

## 实验治理

- Baseline：当前 row-wise `ridge_mse` 与通用 fallback。
- Direct comparison：lagged ridge、LSTM、GRU 在同一特征集合和同一 train/test split 下的预测差异。
- Role：diagnose，验证 sequence 输入和 PyTorch artifact 能否进入稳定 `src/model` 契约。
- Target window：2026-01-01 到 2026-06-09，2026-04-30 前训练，之后测试。
- Changed layer：model。
- External reference：PyTorch 官方 `nn.LSTM` 与 `nn.GRU` batch-first recurrent module 文档；本轮只使用最小 recurrent regressor，不宣称完整论文级 ALSTM/Transformer。
- Acceptance：三个 sequence 模型输出 `paper_strict_adapter`；lagged ridge 区别于 row-wise ridge；LSTM/GRU 预测不同；GRU artifact 保存/加载后保留 adapter metadata。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').exists() and (path / 'AGENTS.md').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact, load_model_artifact, save_model_artifact
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_predict_model, model_adapter_name, model_adapter_status

CARD = repo_root / 'research' / 'sequence_adapter_batch_v1'
CARD.mkdir(parents=True, exist_ok=True)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
test = dataset.frame.loc[dataset.frame['date'] > '2026-04-30'].copy()
feature_columns = dataset.feature_columns[:5]
{
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'train_end': str(train['date'].max().date()),
    'test_start': str(test['date'].min().date()),
    'test_end': str(test['date'].max().date()),
}


{'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d'],
 'train_end': '2026-04-30',
 'test_start': '2026-05-01',
 'test_end': '2026-06-08'}

In [2]:
model_names = ('ridge_mse', 'sequence_lag_ridge_mse', 'lstm_mse', 'gru_mse')
expected = {
    'ridge_mse': ('paper_strict_adapter', 'ridge_mse_v1'),
    'sequence_lag_ridge_mse': ('paper_strict_adapter', 'sequence_lag_ridge_mse_v1'),
    'lstm_mse': ('paper_strict_adapter', 'lstm_pytorch_sequence_mse_v1'),
    'gru_mse': ('paper_strict_adapter', 'gru_pytorch_sequence_mse_v1'),
}

predictions = pd.concat(
    [fit_predict_model(name, train=train, test=test, feature_columns=feature_columns) for name in model_names],
    ignore_index=True,
)
predictions.to_csv(CARD / 'adapter_predictions.csv', index=False)

for name, (status, adapter_name) in expected.items():
    subset = predictions.loc[predictions['model'] == name]
    assert subset['adapter_status'].unique().tolist() == [status]
    assert subset['model_adapter'].unique().tolist() == [adapter_name]
    assert model_adapter_status(name) == status
    assert model_adapter_name(name) == adapter_name

wide = predictions.pivot(index='date', columns='model', values='prediction').reset_index()
wide['sequence_lag_minus_ridge'] = wide['sequence_lag_ridge_mse'] - wide['ridge_mse']
wide['gru_minus_lstm'] = wide['gru_mse'] - wide['lstm_mse']
wide.to_csv(CARD / 'sequence_prediction_deltas.csv', index=False)

assert not wide['sequence_lag_ridge_mse'].round(10).equals(wide['ridge_mse'].round(10))
assert not wide['gru_mse'].round(10).equals(wide['lstm_mse'].round(10))
wide.head()


model,date,gru_mse,lstm_mse,ridge_mse,sequence_lag_ridge_mse,sequence_lag_minus_ridge,gru_minus_lstm
0,2026-05-01,-0.024495,-0.017306,0.010573,-0.011609,-0.022181,-0.007190
1,2026-05-04,-0.011807,-0.012040,0.011120,0.000714,-0.010406,0.000233
2,2026-05-05,0.021667,0.024647,0.023887,0.047276,0.023390,-0.002979
3,2026-05-06,0.021716,0.044483,0.028728,0.024165,-0.004563,-0.022768
4,2026-05-07,0.008813,0.033174,0.016620,-0.000049,-0.016669,-0.024362


In [3]:
summary_rows = []
group_columns = ['model', 'objective', 'family', 'adapter_status', 'model_adapter']
for keys, group in predictions.groupby(group_columns, dropna=False):
    summary_rows.append(
        {
            **dict(zip(group_columns, keys)),
            'rows': int(len(group)),
            'prediction_mean': float(group['prediction'].mean()),
            'prediction_std': float(group['prediction'].std(ddof=0)),
            'target_corr': float(group['prediction'].corr(group['target_return'])),
        }
    )
adapter_summary = pd.DataFrame(summary_rows)
adapter_summary.to_csv(CARD / 'adapter_summary.csv', index=False)
adapter_summary


,model,objective,family,adapter_status,model_adapter,rows,prediction_mean,prediction_std,target_corr
0,gru_mse,mse,sequence,paper_strict_adapter,gru_pytorch_sequence_mse_v1,26,0.028777,0.023204,-0.141275
1,lstm_mse,mse,sequence,paper_strict_adapter,lstm_pytorch_sequence_mse_v1,26,0.031736,0.020347,-0.206792
2,ridge_mse,mse,linear,paper_strict_adapter,ridge_mse_v1,26,0.016191,0.009502,-0.179553
3,sequence_lag_ridge_mse,mse,sequence,paper_strict_adapter,sequence_lag_ridge_mse_v1,26,0.016890,0.020100,0.027226


In [4]:
artifact = fit_model_artifact('gru_mse', train=train, feature_columns=feature_columns)
artifact_path = save_model_artifact(CARD / 'gru_mse_artifact.joblib', artifact)
loaded = load_model_artifact(artifact_path)
artifact_predictions = loaded.predict(test)
artifact_metadata = loaded.metadata_payload()
artifact_predictions.to_csv(CARD / 'gru_artifact_predictions.csv', index=False)
(CARD / 'gru_artifact_metadata.json').write_text(
    json.dumps(artifact_metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

assert loaded.adapter_status == 'paper_strict_adapter'
assert artifact_metadata['model_adapter'] == 'gru_pytorch_sequence_mse_v1'
assert artifact_predictions['model_adapter'].unique().tolist() == ['gru_pytorch_sequence_mse_v1']
{
    'artifact_path': str(artifact_path.relative_to(CARD)),
    'metadata': artifact_metadata,
    'prediction_rows': int(len(artifact_predictions)),
}


{'artifact_path': 'gru_mse_artifact.joblib',
 'metadata': {'schema_version': 'model_artifact.v1',
  'model_name': 'gru_mse',
  'feature_columns': ['soxl_return_1d',
   'soxx_return_1d',
   'soxl_return_5d',
   'soxx_return_5d',
   'soxl_soxx_relative_5d'],
  'objective': 'mse',
  'family': 'sequence',
  'adapter_status': 'paper_strict_adapter',
  'model_adapter': 'gru_pytorch_sequence_mse_v1',
  'target_column': 'target_return',
  'train_rows': 82},
 'prediction_rows': 26}

In [5]:
matrix = run_quantbench_matrix_smoke(
    card_dir=CARD / 'matrix_card',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=model_names,
)
manifest = json.loads((CARD / 'matrix_card' / 'artifact_manifest.json').read_text(encoding='utf-8'))
assert manifest['metadata']['adapter_status'] == 'paper_strict_adapter'
assert manifest['metadata']['adapter_statuses'] == ['paper_strict_adapter']
matrix.model_comparison.to_csv(CARD / 'matrix_model_comparison.csv', index=False)
{
    'matrix_status': matrix.summary['status'],
    'top_model': matrix.summary['top_model'],
    'adapter_status': manifest['metadata']['adapter_status'],
    'model_adapters': manifest['metadata']['model_adapters'],
    'manifest_hash': manifest['manifest_hash'],
}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'matrix_status': 'smoke_completed',
 'top_model': 'sequence_lag_ridge_mse',
 'adapter_status': 'paper_strict_adapter',
 'model_adapters': ['gru_pytorch_sequence_mse_v1',
  'lstm_pytorch_sequence_mse_v1',
  'ridge_mse_v1',
  'sequence_lag_ridge_mse_v1'],
 'manifest_hash': '786266b48f432b7fc79ab483611335b96e9ceda0ec6e3f3d74f3cc3e3295df12'}

In [6]:
analysis_summary = {
    'schema_version': 'sequence_adapter_batch_review.v1',
    'status': 'completed',
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'models': list(model_names),
    'adapter_statuses': sorted(predictions['adapter_status'].unique().tolist()),
    'model_adapters': sorted(predictions['model_adapter'].unique().tolist()),
    'sequence_lag_vs_ridge_abs_mean_diff': float(wide['sequence_lag_minus_ridge'].abs().mean()),
    'gru_vs_lstm_abs_mean_diff': float(wide['gru_minus_lstm'].abs().mean()),
    'adapter_summary_path': 'adapter_summary.csv',
    'prediction_path': 'adapter_predictions.csv',
    'delta_path': 'sequence_prediction_deltas.csv',
    'artifact_path': 'gru_mse_artifact.joblib',
    'matrix_manifest': 'matrix_card/artifact_manifest.json',
    'matrix_top_model': matrix.summary['top_model'],
    'matrix_manifest_hash': manifest['manifest_hash'],
    'external_references': [
        'https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html',
        'https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html',
    ],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2), encoding='utf-8')
analysis_summary


{'schema_version': 'sequence_adapter_batch_review.v1',
 'status': 'completed',
 'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d'],
 'models': ['ridge_mse', 'sequence_lag_ridge_mse', 'lstm_mse', 'gru_mse'],
 'adapter_statuses': ['paper_strict_adapter'],
 'model_adapters': ['gru_pytorch_sequence_mse_v1',
  'lstm_pytorch_sequence_mse_v1',
  'ridge_mse_v1',
  'sequence_lag_ridge_mse_v1'],
 'sequence_lag_vs_ridge_abs_mean_diff': 0.016257825085778714,
 'gru_vs_lstm_abs_mean_diff': 0.01026967679780094,
 'adapter_summary_path': 'adapter_summary.csv',
 'prediction_path': 'adapter_predictions.csv',
 'delta_path': 'sequence_prediction_deltas.csv',
 'artifact_path': 'gru_mse_artifact.joblib',
 'matrix_manifest': 'matrix_card/artifact_manifest.json',
 'matrix_top_model': 'sequence_lag_ridge_mse',
 'matrix_manifest_hash': '786266b48f432b7fc79ab483611335b96e9ceda0ec6e3f3d74f3cc3e3295df12',


## 中文结论

- 做了什么：新增并验证 `sequence_lag_ridge_mse`、`lstm_mse`、`gru_mse` 三个 sequence adapter；序列输入使用固定 lookback 窗口，LSTM/GRU 使用 PyTorch recurrent module 的 batch-first 输入契约。
- 核心指标：四个模型的 adapter 状态、预测均值/波动、预测与目标相关性写入 `adapter_summary.csv`；lagged ridge 与 row-wise ridge 的平均绝对差异、GRU 与 LSTM 的平均绝对差异写入 `analysis_summary.json`。
- regime 差异：本轮小矩阵通过 `matrix_card` 写出 `regime_metrics.csv`、`window_metrics.csv`、`qlib_risk_metrics.csv` 和 `model_effect_deltas.csv`，证明 sequence adapter 能进入已有 qlib/regime 分析链路。
- 是否晋升：否。本轮只是 sequence 模型入口和 artifact 契约验证，不代表生产策略晋升。
- 风险：当前 LSTM/GRU 是最小可训练 recurrent regressor，尚未覆盖 ALSTM/SFM/DA-RNN/TCN/Transformer 的论文级结构，也未做长窗口调参。
- 下一步：继续把 ALSTM/SFM/DA-RNN/TCN/DLinear/PatchTST/Autoformer/FEDformer 分批接入，并在同一 manifest/dashboard/analysis 契约下比较 regime 表现。